In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import rasterio
import re
import tifffile
from keras.models import load_model
from keras.preprocessing.image import img_to_array



In [ ]:
# landsat 7 images -> used interpolation technique

In [ ]:
import os
import re
import numpy as np
import rasterio
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
import scipy.ndimage as ndimage

size = 256
model = load_model('forest_detection_model_vgg16_unet_311223.h5')

def scaleStd(x):
    return (x - (np.nanmean(x) - np.nanstd(x) * 2)) / (
        (np.nanmean(x) + np.nanstd(x) * 2) - (np.nanmean(x) - np.nanstd(x) * 2))

def preprocess_input(image):
    image = image / 255.0
    return image

def fill_gaps_with_interpolation(band):
    if np.ma.isMaskedArray(band):
        band = band.filled(np.nan)
    mask = np.isnan(band)
    filled_band = ndimage.generic_filter(band, np.nanmean, size=3)
    band[mask] = filled_band[mask]
    return band

def calculate_deforestation_per_year(image_paths, tile_size=size):
    deforestation_percentages = []
    forest_percentages = []

    for image_path in image_paths:
        tiles = preprocess_image(image_path, tile_size)
        predictions = [model.predict(np.expand_dims(tile, axis=0)) for tile in tiles]

        deforestation_percentages_i = []
        for prediction in predictions:
            sum_prediction = np.sum(prediction, axis=-1)
            non_zero_mask = sum_prediction != 0
            deforestation_percentage = np.zeros_like(sum_prediction)
            deforestation_percentage[non_zero_mask] = (
                sum_prediction[non_zero_mask] / sum_prediction[non_zero_mask].max()) * 100
            deforestation_percentages_i.append(np.nanmean(deforestation_percentage))

        avg_deforestation = np.nanmean(deforestation_percentages_i)
        deforestation_percentages.append(avg_deforestation)
        forest_percentages.append(100 - avg_deforestation)

    return deforestation_percentages, forest_percentages

def visualize_deforestation(image_paths):
    deforestation_percentages, forest_percentages = calculate_deforestation_per_year(image_paths)

    for i, image_path in enumerate(image_paths):
        year = extract_year_from_filename(os.path.basename(image_path))
        print(f"Non-Forest Percentage - {year}: {deforestation_percentages[i]:.2f}%")
        print(f"Remaining Forest Percentage - {year}: {forest_percentages[i]:.2f}%")

def extract_year_from_filename(filename):
    match = re.search(r'\d{4}', filename)
    return int(match.group()) if match else None

def preprocess_image(image_path, tile_size=size):
    with rasterio.open(image_path) as src:
        red = fill_gaps_with_interpolation(src.read(1, masked=True))
        green = fill_gaps_with_interpolation(src.read(2, masked=True))
        blue = fill_gaps_with_interpolation(src.read(3, masked=True))

    r_std = scaleStd(red)
    g_std = scaleStd(green)
    b_std = scaleStd(blue)

    rgb_std = np.dstack((r_std, g_std, b_std))
    rgb_std = np.clip(rgb_std, 0, 1)
    rgb_std = (rgb_std * 255).astype(np.uint8)

    img_height, img_width, _ = rgb_std.shape

    tiles = []
    for y in range(0, img_height, tile_size):
        for x in range(0, img_width, tile_size):
            tile = rgb_std[y:y + tile_size, x:x + tile_size, :]
            tiles.append(tile)

    max_width = max(tile.shape[1] for tile in tiles)
    max_height = max(tile.shape[0] for tile in tiles)
    padded_tiles = [
        np.pad(tile, ((0, max_height - tile.shape[0]), (0, max_width - tile.shape[1]), (0, 0)), 'constant')
        for tile in tiles
    ]
    preprocessed_tiles = [preprocess_input(img_to_array(tile)) for tile in padded_tiles]

    return preprocessed_tiles

# Directory of normalized Landsat images (one per year)
image_directory = "testimage/tang/landsat_7_interpolation"
image_paths = [os.path.join(image_directory, f) for f in os.listdir(image_directory) if f.endswith(".tif")]

# Run deforestation analysis
visualize_deforestation(image_paths)


In [ ]:
# landsat 8 images normalized 

In [ ]:
import tensorflow as tf
size = 256
# Load the pre-trained U-Net model
model = load_model('forest_detection_model_vgg16_unet_311223.h5')

def scaleStd(x):
    return (x - (np.nanmean(x) - np.nanstd(x) * 2)) / ((np.nanmean(x) + np.nanstd(x) * 2) - (np.nanmean(x) - np.nanstd(x) * 2))

def preprocess_input(image):
    # Assuming pixel values are in the range [0, 255]
    image = image / 255.0
    return image

def calculate_deforestation_per_year(image_paths, tile_size=size):
    num_years = len(image_paths)
    
    deforestation_percentages = []
    forest_percentages = []
    
    for i in range(num_years):
        # Preprocess the image and split into tiles
        tiles = preprocess_image(image_paths[i], tile_size)

        # Predict using the trained model for all tiles
        predictions = [model.predict(np.expand_dims(tile, axis=0)) for tile in tiles]

        # Calculate the percentage of non-forest for each tile
        deforestation_percentages_i = []
        for prediction in predictions:
            sum_prediction = np.sum(prediction, axis=-1)
            
            # Calculate the percentage of non-forest (deforestation)
            non_zero_mask = sum_prediction != 0
            deforestation_percentage = np.zeros_like(sum_prediction)
            deforestation_percentage[non_zero_mask] = (
                sum_prediction[non_zero_mask] / sum_prediction[non_zero_mask].max()
            ) * 100

            deforestation_percentages_i.append(np.nanmean(deforestation_percentage))

        # Calculate overall deforestation and forest percentage for the year
        avg_deforestation = np.nanmean(deforestation_percentages_i)
        deforestation_percentages.append(avg_deforestation)
        forest_percentages.append(100 - avg_deforestation)

    return deforestation_percentages, forest_percentages


def visualize_deforestation(image_paths):
    deforestation_percentages, forest_percentages = calculate_deforestation_per_year(image_paths)

    for i, image_path in enumerate(image_paths):
        year = extract_year_from_filename(os.path.basename(image_path))

        deforestation_text = f"Non-Forest Percentage - {year}: {deforestation_percentages[i]:.2f}%"
        forest_text = f"Remaining Forest Percentage - {year}: {forest_percentages[i]:.2f}%"
        
        print(deforestation_text)
        print(forest_text)


def extract_year_from_filename(filename):
    # Extract the last 4 digits from the filename
    match = re.search(r'\d{4}', filename)
    if match:
        return int(match.group())
    else:
        return None
    
def preprocess_image(image_path, tile_size=size):
    # Load the image using rasterio for specific band access
    with rasterio.open(image_path) as src:
        red = src.read(1, masked=True)
        green = src.read(2, masked=True)
        blue = src.read(3, masked=True)

    # Normalize bands
    r_std = scaleStd(red)
    g_std = scaleStd(green)
    b_std = scaleStd(blue)

    # Stack normalized bands into RGB image
    rgb_std = np.dstack((r_std, g_std, b_std))

    # Convert to float32 and scale to 0–255
    rgb_std = np.clip(rgb_std, 0, 1)  # Make sure values are within range
    rgb_std = (rgb_std * 255).astype(np.uint8)

    img_height, img_width, _ = rgb_std.shape

    # Split into tiles
    tiles = []
    for y in range(0, img_height, tile_size):
        for x in range(0, img_width, tile_size):
            tile = rgb_std[y:y+tile_size, x:x+tile_size, :]
            tiles.append(tile)

    # Pad tiles with zeros to match maximum height/width
    max_width = max(tile.shape[1] for tile in tiles)
    max_height = max(tile.shape[0] for tile in tiles)
    padded_tiles = [
        np.pad(tile, ((0, max_height - tile.shape[0]), (0, max_width - tile.shape[1]), (0, 0)), 'constant')
        for tile in tiles
    ]

    # Preprocess for model input
    preprocessed_tiles = [preprocess_input(img_to_array(tile)) for tile in padded_tiles]

    return preprocessed_tiles


# Provide the directory containing satellite images for different years
image_directory = "testimage/tang/landsat_8_normalized/2013-2023"
image_paths = [os.path.join(image_directory, filename) for filename in os.listdir(image_directory) if filename.endswith(".tif")]

# Visualize deforestation on the original images without showing the map
visualize_deforestation(image_paths)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data preparation
years = np.arange(2006, 2026)
non_forest = np.array([
    78.82, 78.58, 78.54, 77.81, 78.50, 78.49, 77.56,
    79.93, 81.22, 81.83, 80.89, 81.29, 80.75,
    81.31, 81.04, 81.19, 81.13, 81.09, 81.08, 81.58
])
forest = np.array([
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
])

# Create figure with dual y-axis
fig, ax1 = plt.subplots(figsize=(14, 7))

# Forest cover (left axis)
ax1.plot(years, forest, 'g-o', linewidth=2, markersize=8, label='Forest Cover (%)')
ax1.set_ylabel('Forest Cover Percentage', color='g', fontsize=12)
ax1.tick_params(axis='y', labelcolor='g')
ax1.set_ylim(15, 25)  # Adjusted to show detail

# Non-forest (right axis)
ax2 = ax1.twinx()
ax2.plot(years, non_forest, 'r-s', linewidth=2, markersize=8, label='Non-Forest (%)')
ax2.set_ylabel('Non-Forest Percentage', color='r', fontsize=12)
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(75, 85)  # Mirrors forest scale

# Formatting
ax1.set_title('Modhupur Forest Cover Change (2006-2025)', fontsize=14, pad=20)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_xticks(years)
ax1.set_xticklabels(years, rotation=45, ha='right')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# Highlight key changes
ax1.axvspan(2012, 2013, color='yellow', alpha=0.3, label='Sharp Decline')
ax1.axvspan(2024, 2025, color='orange', alpha=0.3, label='Recent Drop')

# Grid and layout
ax1.grid(True, linestyle='--', alpha=0.7)
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Data
years = list(range(2006, 2026))
forest_percentage = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25, 18.69,
    18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]
non_forest_percentage = [
    78.82, 78.58, 78.54, 77.81, 78.50, 78.49, 77.56,
    79.93, 81.22, 81.83, 80.89, 81.29, 80.75, 81.31,
    81.04, 81.19, 81.13, 81.09, 81.08, 81.58
]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(years, forest_percentage, marker='o', label='Remaining Forest (%)', color='green')
plt.plot(years, non_forest_percentage, marker='o', label='Non-Forest (%)', color='brown')

plt.title('Modhupur Forest Area (2006-2025)\nForest vs Non-Forest Percentage')
plt.xlabel('Year')
plt.ylabel('Percentage (%)')
plt.ylim(0, 100)
plt.xticks(years)  # <== This will show all years on X-axis
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# -------------------------------
# Clay2 style
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 18
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Data
# -------------------------------
years = [str(y) for y in range(2006, 2026)]


remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25, 18.69,
    18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]
non_forest_percentages = [
    78.82, 78.58, 78.54, 77.81, 78.50, 78.49, 77.56,
    79.93, 81.22, 81.83, 80.89, 81.29, 80.75, 81.31,
    81.04, 81.19, 81.13, 81.09, 81.08, 81.58
]

# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

# Plot lines
ax.plot(years, non_forest_percentages, '-o', color='red', label='Non-Forest %', markersize=6)
ax.plot(years, remaining_forest_percentages, '-o', color='green', label='Remaining Forest %', markersize=6)

# Annotate all points
for x, y in zip(years, remaining_forest_percentages):
    ax.text(x, y - 0.25, f'{y:.2f}', ha='center', va='top', fontsize=9, color='green')

for x, y in zip(years, non_forest_percentages):
    ax.text(x, y + 0.25, f'{y:.2f}', ha='center', va='bottom', fontsize=9, color='red')

# Labels and title
ax.set_title('Modhupur Deforestation Trend (2006–2025)', fontsize=20)
ax.set_xlabel('Years', fontsize=18)
ax.set_ylabel('Percentage (%)', fontsize=18)

# Ticks styling
#ax.tick_params(axis='both', which='major', direction='out', length=6, width=2, colors='black', labelsize=14)
#ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2, colors='black', labelsize=14)
ax.xaxis.set_minor_locator(plt.MultipleLocator(1))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.5))

# Thicken axes
for spine in ax.spines.values():
    spine.set_linewidth(1)

# Legend
ax.legend(frameon=False, fontsize=16)

# Remove grid (Clay2 style)
ax.grid(False)

# X-axis rotation
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# -------------------------------
# Data
# -------------------------------
years = [
    '2006', '2007', '2008', '2009', '2010', '2011', '2012',
    '2013', '2014', '2015', '2016', '2017', '2018',
    '2019', '2020', '2021', '2022', '2023', '2024', '2025'
]


remaining_forest = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25, 18.69,
    18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]

# Create time series
forest_series = pd.Series(remaining_forest, index=pd.to_datetime(years))

# Manual forecast values
forecast_dates = pd.to_datetime(['2024', '2025'])
forecast = [18.20, 18.44]

# Create comparison table
results = pd.DataFrame({
    'Year': ['2024', '2025'],
    'Actual (%)': [18.38, 18.03],
    'Forecast (%)': forecast,
    'Error (%)': (abs(pd.Series(forecast) - [18.38, 18.03]) / [18.38, 18.03] * 100).round(1)
})

# -------------------------------
# Clay2 Template Global Settings
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 25
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Create figure + axis
# -------------------------------
fig, ax = plt.subplots(figsize=(10, 7))

# Actual data (blue)
ax.plot(forest_series.index, forest_series, 'bo-', markersize=8, label='Actual Forest Cover')

# Forecast data (red)
#ax.plot(forecast_dates, forecast, 'ro--', markersize=8, label='ARIMA Forecast')

# Data labels (smaller font)
for date, value in zip(forest_series.index, forest_series):
    ax.text(date, value-0.20, f'{value:.2f}%', ha='center', va='top',
            color='blue', fontsize=10)



# Vertical line for forecast start with legend
#ax.axvline(x=pd.to_datetime('2024'), linestyle='--', color='black', linewidth=2, label='Forecast Start')

# Axis labels
ax.set_xlabel('Year', fontsize=25)
ax.set_ylabel('Forest Cover (%)', fontsize=25)

# Customize ticks (clay style)
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2,
               colors='black', labelsize=20)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2,
               colors='black', labelsize=20)

# X-axis: every 2 years, rotated
xticks = forest_series.index[::2]
ax.set_xticks(xticks)
ax.set_xticklabels([d.year for d in xticks], rotation=45, fontsize=20)

# Y ticks
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.25))
ax.minorticks_on()

# Thicken axis lines
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=18, loc='upper right')

plt.tight_layout()
plt.show()

# Print results
print("\n=== Manual Forecast Comparison ===")
print(results.to_string(index=False))

mape = results['Error (%)'].mean()
print(f"\nMean Absolute Percentage Error (MAPE): {mape:.1f}%")


In [ ]:
import matplotlib.pyplot as plt

# Years updated to match the data length
years = [
    '2006', '2007', '2008', '2009', '2010', '2011', '2012',
    '2013', '2014', '2015', '2016', '2017', '2018',
    '2019', '2020', '2021', '2022', '2023'
]

non_forest_percentages = [
    78.82, 78.58, 78.54, 77.81, 78.50, 78.49, 77.56,
    79.93, 81.22, 81.83, 80.89, 81.29, 80.75,
    81.31, 81.04, 81.19, 81.13, 81.09
]

remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91
]



# Plotting
plt.figure(figsize=(12, 6))
plt.plot(years, non_forest_percentages, marker='o', label='Non-Forest Percentage', color='red')
plt.plot(years, remaining_forest_percentages, marker='o', label='Remaining Forest Percentage', color='green')

plt.title('modhupur Deforestation Trend (2006–2023)', fontsize=14)
plt.xlabel('Years')
plt.ylabel('Percentage')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Prepare the time series data
years = list(range(2006, 2024))
non_forest_percentages = [
    78.82, 78.58, 78.54, 77.81, 78.50, 78.49, 77.56,
    79.93, 81.22, 81.83, 80.89, 81.29, 80.75,
    81.31, 81.04, 81.19, 81.13, 81.09
]

# Convert to pandas Series
data = pd.Series(non_forest_percentages, index=pd.Index(years, name="Year"))

# Fit ARIMA model (auto order tuning can be done with pmdarima, but we manually pick for now)
model = ARIMA(data, order=(1, 1, 1))  # (p, d, q)
model_fit = model.fit()

# Forecast for next 2 years (2024 & 2025)
forecast = model_fit.forecast(steps=2)
forecast_years = [2024, 2025]

# Combine with original data for plotting
all_years = years + forecast_years
all_values = non_forest_percentages + list(forecast)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(years, non_forest_percentages, label="Historical", marker='o', color='red')
plt.plot(forecast_years, forecast, label="Forecast", marker='o', linestyle='--', color='blue')
plt.title("Forecast of Non-Forest Percentage (ARIMA)")
plt.xlabel("Year")
plt.ylabel("Non-Forest Percentage")
plt.xticks(all_years, rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast values
for year, value in zip(forecast_years, forecast):
    print(f"Forecasted Non-Forest Percentage for {year}: {value:.2f}%")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Prepare the time series data
years = list(range(2006, 2024))
remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91
]

# Convert to pandas Series
data = pd.Series(remaining_forest_percentages, index=pd.Index(years, name="Year"))

# Fit ARIMA model (auto order tuning can be done with pmdarima, but we manually pick for now)
model = ARIMA(data, order=(1, 1, 1))  # (p, d, q)
model_fit = model.fit()

# Forecast for next 2 years (2024 & 2025)
forecast = model_fit.forecast(steps=2)
forecast_years = [2024, 2025]

# Combine with original data for plotting
all_years = years + forecast_years
all_values = non_forest_percentages + list(forecast)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(years, remaining_forest_percentages, label="Historical", marker='o', color='red')
plt.plot(forecast_years, forecast, label="Forecast", marker='o', linestyle='--', color='blue')
plt.title("Forecast of Forest Percentage (ARIMA)")
plt.xlabel("Year")
plt.ylabel("Non-Forest Percentage")
plt.xticks(all_years, rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast values
for year, value in zip(forecast_years, forecast):
    print(f"Forecasted Forest Percentage for {year}: {value:.2f}%")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Historical data
years = list(range(2006, 2024))

non_forest_percentages = [
    78.82, 78.58, 78.54, 77.81, 78.50, 78.49, 77.56,
    79.93, 81.22, 81.83, 80.89, 81.29, 80.75,
    81.31, 81.04, 81.19, 81.13, 81.09
]

# Convert to time series
data = pd.Series(non_forest_percentages, index=pd.Index(years, name="Year"))

# Fit ARIMA model
model = ARIMA(data, order=(1, 1, 1))  # You can tune this
model_fit = model.fit()

# Forecast for 2024 and 2025
forecast_steps = 2
forecast_years = [2024, 2025]
forecast_non_forest = model_fit.forecast(steps=forecast_steps)
forecast_forest = 100 - forecast_non_forest

# Combine full time series
all_years = years + forecast_years
all_non_forest = non_forest_percentages + list(forecast_non_forest)
all_forest = [100 - val for val in all_non_forest]

# Plot both
plt.figure(figsize=(12, 6))
plt.plot(all_years, all_non_forest, marker='o', label="Non-Forest %", color='red')
plt.plot(all_years, all_forest, marker='o', label="Forest %", color='green')
plt.axvline(x=2023.5, color='gray', linestyle='--', label='Forecast Starts')
plt.title("Forecast of Forest and Non-Forest Percentage (2024–2025)")
plt.xlabel("Year")
plt.ylabel("Percentage")
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.xticks(all_years, rotation=45)
plt.tight_layout()
plt.show()

# Print forecasted values


for i in range(forecast_steps):
    print(f"Year {forecast_years[i]} - Forecasted:")
    print(f"  Non-Forest Percentage: {forecast_non_forest.iloc[i]:.2f}%")
    print(f"  Forest Percentage: {forecast_forest.iloc[i]:.2f}%")


In [ ]:
import tensorflow as tf
size = 256
# Load the pre-trained U-Net model
model = load_model('forest_detection_model_vgg16_unet_311225.h5')

def scaleStd(x):
    return (x - (np.nanmean(x) - np.nanstd(x) * 2)) / ((np.nanmean(x) + np.nanstd(x) * 2) - (np.nanmean(x) - np.nanstd(x) * 2))

def preprocess_input(image):
    # Assuming pixel values are in the range [0, 255]
    image = image / 255.0
    return image

def calculate_deforestation_per_year(image_paths, tile_size=size):
    num_years = len(image_paths)
    
    deforestation_percentages = []
    forest_percentages = []
    
    for i in range(num_years):
        # Preprocess the image and split into tiles
        tiles = preprocess_image(image_paths[i], tile_size)

        # Predict using the trained model for all tiles
        predictions = [model.predict(np.expand_dims(tile, axis=0)) for tile in tiles]

        # Calculate the percentage of non-forest for each tile
        deforestation_percentages_i = []
        for prediction in predictions:
            sum_prediction = np.sum(prediction, axis=-1)
            
            # Calculate the percentage of non-forest (deforestation)
            non_zero_mask = sum_prediction != 0
            deforestation_percentage = np.zeros_like(sum_prediction)
            deforestation_percentage[non_zero_mask] = (
                sum_prediction[non_zero_mask] / sum_prediction[non_zero_mask].max()
            ) * 100

            deforestation_percentages_i.append(np.nanmean(deforestation_percentage))

        # Calculate overall deforestation and forest percentage for the year
        avg_deforestation = np.nanmean(deforestation_percentages_i)
        deforestation_percentages.append(avg_deforestation)
        forest_percentages.append(100 - avg_deforestation)

    return deforestation_percentages, forest_percentages


def visualize_deforestation(image_paths):
    deforestation_percentages, forest_percentages = calculate_deforestation_per_year(image_paths)

    for i, image_path in enumerate(image_paths):
        year = extract_year_from_filename(os.path.basename(image_path))

        deforestation_text = f"Non-Forest Percentage - {year}: {deforestation_percentages[i]:.2f}%"
        forest_text = f"Remaining Forest Percentage - {year}: {forest_percentages[i]:.2f}%"
        
        print(deforestation_text)
        print(forest_text)


def extract_year_from_filename(filename):
    # Extract the last 4 digits from the filename
    match = re.search(r'\d{4}', filename)
    if match:
        return int(match.group())
    else:
        return None
    
def preprocess_image(image_path, tile_size=size):
    # Load the image using rasterio for specific band access
    with rasterio.open(image_path) as src:
        red = src.read(1, masked=True)
        green = src.read(2, masked=True)
        blue = src.read(3, masked=True)

    # Normalize bands
    r_std = scaleStd(red)
    g_std = scaleStd(green)
    b_std = scaleStd(blue)

    # Stack normalized bands into RGB image
    rgb_std = np.dstack((r_std, g_std, b_std))

    # Convert to float32 and scale to 0–255
    rgb_std = np.clip(rgb_std, 0, 1)  # Make sure values are within range
    rgb_std = (rgb_std * 255).astype(np.uint8)

    img_height, img_width, _ = rgb_std.shape

    # Split into tiles
    tiles = []
    for y in range(0, img_height, tile_size):
        for x in range(0, img_width, tile_size):
            tile = rgb_std[y:y+tile_size, x:x+tile_size, :]
            tiles.append(tile)

    # Pad tiles with zeros to match maximum height/width
    max_width = max(tile.shape[1] for tile in tiles)
    max_height = max(tile.shape[0] for tile in tiles)
    padded_tiles = [
        np.pad(tile, ((0, max_height - tile.shape[0]), (0, max_width - tile.shape[1]), (0, 0)), 'constant')
        for tile in tiles
    ]

    # Preprocess for model input
    preprocessed_tiles = [preprocess_input(img_to_array(tile)) for tile in padded_tiles]

    return preprocessed_tiles


# Provide the directory containing satellite images for different years
image_directory = "testimage/tang/landsat_8_normalized/2024-2025"
image_paths = [os.path.join(image_directory, filename) for filename in os.listdir(image_directory) if filename.endswith(".tif")]

# Visualize deforestation on the original images without showing the map
visualize_deforestation(image_paths)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Forest percentage data from 2006 to 2023
years = list(range(2006, 2024))
forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91
]

# Create a pandas Series
forest_series = pd.Series(forest_percentages, index=pd.Index(years, name="Year"))

# Plot with proper ticks
plt.figure(figsize=(10, 5))
forest_series.plot(title="Remaining Forest Percentage (2006–2023)", marker='o')
plt.ylabel("Forest Percentage")
plt.xticks(ticks=years, rotation=45)  # Ensure all years show
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Fit ARIMA model (p=1, d=1, q=1) as a starting point
model = ARIMA(forest_series, order=(1, 1, 1))
model_fit = model.fit()

# Print summary of the model
print(model_fit.summary())


In [ ]:
forecast = model_fit.forecast(steps=2)
forecast_years = list(range(2024, 2026))

# Plot actual and forecasted data
plt.plot(forest_series, label="Historical")
plt.plot(forecast_years, forecast, label="Forecast", marker='o', linestyle='--')
plt.title("Forecast of Forest Percentage (2024–2028)")
plt.ylabel("Forest Percentage")
plt.xlabel("Year")
plt.legend()
plt.grid(True)
plt.show()

# View forecasted values
for year, value in zip(forecast_years, forecast):
    print(f"Forecast for {year}: {value:.2f}%")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

# Historical forest percentages (2006–2023)
years = list(range(2006, 2024))
forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91
]

# Actual values for 2024 and 2025
actual_future_years = [2024, 2025]
actual_future_values = [18.92, 18.42]

# Create series for actual historical
forest_series = pd.Series(forest_percentages, index=pd.Index(years, name="Year"))

# Fit ARIMA model and forecast for 2024, 2025
model = ARIMA(forest_series, order=(1, 1, 1))
model_fit = model.fit()
forecast_values = model_fit.forecast(steps=2)
forecast_years = [2024, 2025]
forecast_series = pd.Series(forecast_values.values, index=forecast_years)

# Combine for plotting
combined_series = pd.concat([forest_series, forecast_series])

# Plot
plt.figure(figsize=(10, 5))
plt.plot(forest_series.index, forest_series.values, marker='o', label='Actual (2006–2023)', color='blue')
plt.plot(forecast_series.index, forecast_series.values, marker='o', linestyle='--', color='orange', label='Forecasted (2024–2025)')
plt.plot(actual_future_years, actual_future_values, marker='o', linestyle='-', color='green', label='Actual (2024–2025)')

# Highlight forecast area
plt.axvline(x=2023, linestyle='--', color='gray', label='Forecast Start')

plt.title("Remaining Forest Percentage (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print comparison
print("Actual vs Forecasted (2024–2025):")
for year in forecast_years:
    print(f"{year}: Forecast = {forecast_series[year]:.2f}%, Actual = {actual_future_values[year - 2024]:.2f}%")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA

# Data from 2006 to 2023
years = list(range(2006, 2024))
forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91
]

# Actual values for 2024 and 2025
actual_future_years = [2024, 2025]
actual_future_values = [18.92, 18.42]

# Create series
forest_series = pd.Series(forest_percentages, index=pd.Index(years, name="Year"))

# Auto ARIMA to find best (p, d, q)
stepwise_model = auto_arima(
    forest_series, seasonal=False, trace=True,
    error_action='ignore', suppress_warnings=True, stepwise=True
)
print(f"Selected ARIMA order: {stepwise_model.order}")

# Fit best model from auto_arima
model = ARIMA(forest_series, order=stepwise_model.order)
model_fit = model.fit()
forecast = model_fit.forecast(steps=2)
forecast_years = [2024, 2025]
forecast_series = pd.Series(forecast.values, index=forecast_years)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(forest_series.index, forest_series.values, marker='o', label='Actual (2006–2023)', color='blue')
plt.plot(forecast_series.index, forecast_series.values, marker='o', linestyle='--', color='orange', label='Forecasted (2024–2025)')
plt.plot(actual_future_years, actual_future_values, marker='o', linestyle='-', color='green', label='Actual (2024–2025)')

plt.axvline(x=2023, linestyle='--', color='gray', label='Forecast Start')

plt.title("Remaining Forest Percentage (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print comparison
print("\nActual vs Forecasted (2024–2025):")
for year in forecast_years:
    print(f"{year}: Forecast = {forecast_series[year]:.2f}%, Actual = {actual_future_values[year - 2024]:.2f}%")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA

# Forest percentage data from 2006 to 2023
years = list(range(2006, 2024))
forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91
]

# Actual values for 2024 and 2025
actual_future_years = [2024, 2025]
actual_future_values = [18.92, 18.42]

# Create pandas Series for time series modeling
forest_series = pd.Series(forest_percentages, index=pd.Index(years, name="Year"))

# Use auto_arima to find best order (p, d, q)
stepwise_model = auto_arima(
    forest_series, seasonal=False, trace=True,
    error_action='ignore', suppress_warnings=True, stepwise=True
)
print(f"\nSelected ARIMA order: {stepwise_model.order}")

# Fit ARIMA model
model = ARIMA(forest_series, order=stepwise_model.order)
model_fit = model.fit()

# Forecast for 2024 and 2025
forecast_years = [2024, 2025]
forecast = model_fit.forecast(steps=2)
forecast_series = pd.Series(forecast.values, index=forecast_years)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(forest_series.index, forest_series.values, marker='o', color='blue', label='Actual (2006–2023)')
plt.plot(forecast_series.index, forecast_series.values, marker='o', linestyle='--', color='orange', label='Forecasted (2024–2025)')

# Optional: Connect last actual (2023) to first forecast (2024)
plt.plot([2023, 2024], [forest_series[2023], forecast_series[2024]], color='orange', linestyle='--')

# Mark forecast starting point
plt.axvline(x=2023, linestyle='--', color='gray', label='Forecast Start')

plt.title("Remaining Forest Percentage (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print comparison
print("\nActual vs Forecasted (2024–2025):")
for i, year in enumerate(forecast_years):
    forecast_val = forecast_series[year]
    actual_val = actual_future_values[i]
    diff = abs(forecast_val - actual_val)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {forecast_val:.2f}%, Actual = {actual_val:.2f}%, Difference = {diff:.2f}% ({comment})")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA

# Actual forest percentage data from 2006 to 2025
years = list(range(2006, 2026))
forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42  # includes 2024, 2025 actual
]

# Separate actual data for modeling (up to 2023)
training_years = list(range(2006, 2024))
training_data = forest_percentages[:len(training_years)]

# Fit the ARIMA model on training data
forest_series = pd.Series(training_data, index=pd.Index(training_years, name="Year"))
stepwise_model = auto_arima(
    forest_series, seasonal=False, trace=False,
    error_action='ignore', suppress_warnings=True, stepwise=True
)
print(f"Selected ARIMA order: {stepwise_model.order}")

model = ARIMA(forest_series, order=stepwise_model.order)
model_fit = model.fit()

# Forecast for 2024 and 2025
forecast_years = [2024, 2025]
forecast = model_fit.forecast(steps=2)
forecast_series = pd.Series(forecast.values, index=forecast_years)

# Plotting both actual (including 2024–2025) and forecasted
plt.figure(figsize=(10, 5))

# Full actual data including 2024 & 2025
plt.plot(years, forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')

# Forecasted values for 2024–2025
plt.plot(forecast_series.index, forecast_series.values, marker='o', linestyle='--', color='orange', label='Forecasted (2024–2025)')

# Forecast start line
plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

plt.title("Remaining Forest Percentage (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual comparison
print("\nActual vs Forecasted (2024–2025):")
actual_future_values = [18.92, 18.42]
for i, year in enumerate(forecast_years):
    forecast_val = forecast_series[year]
    actual_val = actual_future_values[i]
    diff = abs(forecast_val - actual_val)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {forecast_val:.2f}%, Actual = {actual_val:.2f}%, Difference = {diff:.2f}% ({comment})")


In [ ]:
# Deep ARIMA Analysis for modhupur Forest Cover
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import numpy as np

# Prepare data
years = list(range(2006, 2026))
remaining_forest = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]
actual_2024, actual_2025 = 18.92, 18.42

# Create complete series
full_series = pd.Series(remaining_forest, index=years)
train = full_series[:-2]  # 2006-2023
test = full_series[-2:]   # 2024-2025

# ADF Test for stationarity
def adf_test(series):
    result = adfuller(series, regression='ct')  # Using trend-aware test
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.3f}")
    return result[1] < 0.05

print("=== ADF Test Results ===")
is_stationary = adf_test(train)
print(f"\nStationary? {is_stationary}\n")

# AutoARIMA Model
print("=== AutoARIMA Model Selection ===")
model = auto_arima(train,
                  seasonal=False,
                  test='adf',
                  regression='ct',
                  stepwise=True,
                  trace=True,
                  suppress_warnings=True)

print(f"\nOptimal ARIMA Order: {model.order}")
print(f"AIC: {model.aic():.2f}")

# Forecast
forecast, conf_int = model.predict(n_periods=2, return_conf_int=True)

# Evaluation
errors = forecast - test.values
mape = np.mean(np.abs(errors/test.values))*100

# Visualization
plt.figure(figsize=(12, 6))
plt.xticks(years, rotation=45)

# Plot actual data (2006-2025)
plt.plot(full_series.index, full_series, 'go-', 
         label='Actual Data (2006-2025)', 
         linewidth=2, markersize=8)

# Plot forecast (2024-2025)
plt.plot([2024, 2025], forecast, 'ro--', 
         label=f'ARIMA{model.order} Forecast', 
         linewidth=2, markersize=8)

# Confidence interval
plt.fill_between([2024, 2025], 
                conf_int[:, 0], conf_int[:, 1],
                color='pink', alpha=0.3, 
                label='95% Confidence Interval')

plt.title('Sundarbans Forest Cover: Actual vs Forecast', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Forest Cover (%)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Results Table
results = pd.DataFrame({
    'Year': [2024, 2025],
    'Actual': test.values,
    'Forecast': forecast.round(2),
    'Lower CI': conf_int[:, 0].round(2),
    'Upper CI': conf_int[:, 1].round(2),
    'Error %': (np.abs(errors/test.values)*100).round(1)
})

print("\n=== Forecast Accuracy ===")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.1f}%")
print("\n=== Forecast vs Actual Comparison ===")
print(results.to_string(index=False))

plt.show()

In [ ]:
# Deep ARIMA Analysis for Sundarbans Forest Cover
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import numpy as np

# Prepare data
years = list(range(2006, 2026))
remaining_forest = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]
actual_2024, actual_2025 = 18.92, 18.42

# Create complete series
full_series = pd.Series(remaining_forest, index=years)
train = full_series[:-2]  # 2006-2023
test = full_series[-2:]   # 2024-2025

# ADF Test for stationarity
def adf_test(series):
    result = adfuller(series, regression='ct')  # Using trend-aware test
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.3f}")
    return result[1] < 0.05

print("=== ADF Test Results ===")
is_stationary = adf_test(train)
print(f"\nStationary? {is_stationary}\n")

# AutoARIMA Model
print("=== AutoARIMA Model Selection ===")
model = auto_arima(train,
                  seasonal=False,
                  test='adf',
                  regression='ct',
                  stepwise=True,
                  trace=True,
                  suppress_warnings=True)

print(f"\nOptimal ARIMA Order: {model.order}")
print(f"AIC: {model.aic():.2f}")

# Forecast
forecast, conf_int = model.predict(n_periods=2, return_conf_int=True)

# Evaluation
errors = forecast - test.values
mape = np.mean(np.abs(errors/test.values))*100

# Visualization
plt.figure(figsize=(12, 6))
plt.xticks(years, rotation=45)

# Plot actual data (2006-2025)
plt.plot(full_series.index, full_series, 'go-', 
         label='Actual Data (2006-2025)', 
         linewidth=2, markersize=8)

# Add percentage labels to actual data points
for year, value in zip(full_series.index, full_series):
    plt.text(year, value-0.3, f'{value:.2f}%', 
             ha='center', va='top', color='green', fontsize=9)

# Plot forecast (2024-2025)
plt.plot([2024, 2025], forecast, 'ro--', 
         label=f'ARIMA{model.order} Forecast', 
         linewidth=2, markersize=8)

# Add percentage labels to forecast points
for year, value in zip([2024, 2025], forecast):
    plt.text(year, value+0.3, f'{value:.2f}%', 
             ha='center', va='bottom', color='red', fontsize=9)

# Confidence interval
plt.fill_between([2024, 2025], 
                conf_int[:, 0], conf_int[:, 1],
                color='pink', alpha=0.3, 
                label='95% Confidence Interval')

plt.title('Sundarbans Forest Cover: Actual vs Forecast', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Forest Cover (%)', fontsize=12)
plt.legend(fontsize=10, loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Results Table
results = pd.DataFrame({
    'Year': [2024, 2025],
    'Actual': test.values,
    'Forecast': forecast.round(2),
    'Lower CI': conf_int[:, 0].round(2),
    'Upper CI': conf_int[:, 1].round(2),
    'Error %': (np.abs(errors/test.values)*100).round(1)
})

print("\n=== Forecast Accuracy ===")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.1f}%")
print("\n=== Forecast vs Actual Comparison ===")
print(results.to_string(index=False))

plt.show()

In [ ]:
# Deep ARIMA Analysis for Forest Cover (Clay2 Template)
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# -------------------------------
# Data
# -------------------------------
years = list(range(2006, 2026))
remaining_forest = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]
actual_2024, actual_2025 = 18.92, 18.42

# Create complete series
full_series = pd.Series(remaining_forest, index=years)
train = full_series[:-2]  # 2006-2023
test = full_series[-2:]   # 2024-2025

# -------------------------------
# ADF Test for Stationarity
# -------------------------------
def adf_test(series):
    result = adfuller(series, regression='ct')  # with trend
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.3f}")
    return result[1] < 0.05

print("=== ADF Test Results ===")
is_stationary = adf_test(train)
print(f"\nStationary? {is_stationary}\n")

# -------------------------------
# AutoARIMA Model
# -------------------------------
print("=== AutoARIMA Model Selection ===")
model = auto_arima(train,
                  seasonal=False,
                  test='adf',
                  regression='ct',
                  stepwise=True,
                  trace=True,
                  suppress_warnings=True)

print(f"\nOptimal ARIMA Order: {model.order}")
print(f"AIC: {model.aic():.2f}")

# -------------------------------
# Forecast
# -------------------------------
forecast, conf_int = model.predict(n_periods=2, return_conf_int=True)
errors = forecast - test.values
mape = np.mean(np.abs(errors/test.values)) * 100

# -------------------------------
# Clay2 Global Plot Settings
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 22
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(10, 7))

# Actual data
ax.plot(full_series.index, full_series, 'bo-', markersize=8, label='Actual Forest Cover')

# Forecast data
ax.plot([2024, 2025], forecast, 'ro--', markersize=8, label=f'ARIMA{model.order} Forecast')

# Confidence interval shading
ax.fill_between([2024, 2025], conf_int[:, 0], conf_int[:, 1],
                color='pink', alpha=0.3, label='95% Confidence Interval')

# Data labels
for year, value in zip(full_series.index, full_series):
    ax.text(year, value-0.15, f'{value:.2f}%', ha='center', va='top',
            color='blue', fontsize=10)

for year, value in zip([2024, 2025], forecast):
    ax.text(year, value+0.25, f'{value:.2f}%', ha='center', va='bottom',
            color='red', fontsize=10)

# Forecast start vertical line
ax.axvline(x=2024, linestyle='--', color='black', linewidth=2, label='Forecast Start')

# Axis labels
ax.set_xlabel('Year', fontsize=22)
ax.set_ylabel('Forest Cover (%)', fontsize=22)

# Customize ticks (Clay2 style)
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2,
               colors='black', labelsize=18)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2,
               colors='black', labelsize=18)

# X ticks: every 2 years
xticks = full_series.index[::2]
ax.set_xticks(xticks)
ax.set_xticklabels([str(x) for x in xticks], rotation=45, fontsize=18)

# Y ticks
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.25))
ax.minorticks_on()

# Thicken axes
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=16, loc='upper right')

plt.tight_layout()
plt.show()

# -------------------------------
# Results Table
# -------------------------------
results = pd.DataFrame({
    'Year': [2024, 2025],
    'Actual': test.values,
    'Forecast': forecast.round(2),
    'Lower CI': conf_int[:, 0].round(2),
    'Upper CI': conf_int[:, 1].round(2),
    'Error %': (np.abs(errors/test.values)*100).round(1)
})

print("\n=== Forecast Accuracy ===")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.1f}%")
print("\n=== Forecast vs Actual Comparison ===")
print(results.to_string(index=False))


In [ ]:
# Deep ARIMA Analysis for Forest Cover (Clay3 Template)
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# -------------------------------
# Data
# -------------------------------
years = list(range(2006, 2026))
remaining_forest = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]
actual_2024, actual_2025 = 18.92, 18.42

# Create complete series
full_series = pd.Series(remaining_forest, index=years)
train = full_series[:-2]  # 2006-2023
test = full_series[-2:]   # 2024-2025

# -------------------------------
# ADF Test for Stationarity
# -------------------------------
def adf_test(series):
    result = adfuller(series, regression='ct')  # with trend
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.3f}")
    return result[1] < 0.05

print("=== ADF Test Results ===")
is_stationary = adf_test(train)
print(f"\nStationary? {is_stationary}\n")

# -------------------------------
# AutoARIMA Model
# -------------------------------
print("=== AutoARIMA Model Selection ===")
model = auto_arima(train,
                  seasonal=False,
                  test='adf',
                  regression='ct',
                  stepwise=True,
                  trace=True,
                  suppress_warnings=True)

print(f"\nOptimal ARIMA Order: {model.order}")
print(f"AIC: {model.aic():.2f}")

# -------------------------------
# Forecast
# -------------------------------
forecast, conf_int = model.predict(n_periods=2, return_conf_int=True)
errors = forecast - test.values
mape = np.mean(np.abs(errors/test.values)) * 100

# -------------------------------
# Clay3 Global Plot Settings
# -------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 25
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

# -------------------------------
# Plot
# -------------------------------
fig, ax = plt.subplots(figsize=(10, 7))

# Actual data
ax.plot(full_series.index, full_series, 'bo-', markersize=8, label='Actual Forest Cover')

# Forecast data
ax.plot([2024, 2025], forecast, 'ro--', markersize=8, label=f'ARIMA Forecast')

# Confidence interval shading
ax.fill_between([2024, 2025], conf_int[:, 0], conf_int[:, 1],
                color='pink', alpha=0.3, label='95% Confidence Interval')

# Data labels
for year, value in zip(full_series.index, full_series):
    ax.text(year, value-0.25, f'{value:.2f}%', ha='center', va='top',
            color='blue', fontsize=10)

for year, value in zip([2024, 2025], forecast):
    ax.text(year, value+0.35, f'{value:.2f}%', ha='center', va='bottom',
            color='red', fontsize=10)

# Forecast start vertical line (shares legend with forecast)
ax.axvline(x=2024, linestyle='--', color='black', linewidth=2, label='Forecast Start')

# Axis labels
ax.set_xlabel('Year', fontsize=25)
ax.set_ylabel('Forest Cover (%)', fontsize=25)

# Customize ticks (Clay3 style)
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2,
               colors='black', labelsize=20)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2,
               colors='black', labelsize=20)

# X ticks: every 2 years
xticks = full_series.index[::2]
ax.set_xticks(xticks)
ax.set_xticklabels([str(x) for x in xticks], rotation=45, fontsize=20)

# Y ticks: difference of 2
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(1))
ax.minorticks_on()

# Thicken axes
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=18, loc='upper right')

plt.tight_layout()
plt.show()

# -------------------------------
# Results Table
# -------------------------------
results = pd.DataFrame({
    'Year': [2024, 2025],
    'Actual': test.values,
    'Forecast': forecast.round(2),
    'Lower CI': conf_int[:, 0].round(2),
    'Upper CI': conf_int[:, 1].round(2),
    'Error %': (np.abs(errors/test.values)*100).round(1)
})

print("\n=== Forecast Accuracy ===")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.1f}%")
print("\n=== Forecast vs Actual Comparison ===")
print(results.to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

# Actual data from 2006 to 2025
years = list(range(2006, 2026))
remaining_forest_percentages = [
     21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]



# Training data up to 2023
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# Normalize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# Prepare sequences (lookback = 3)
lookback = 10
X, y = [], []
for i in range(lookback, len(scaled_data)):
    X.append(scaled_data[i - lookback:i, 0])
    y.append(scaled_data[i, 0])
X, y = np.array(X), np.array(y)
X = X.reshape((X.shape[0], X.shape[1], 1))

# Build LSTM model
model = Sequential([
    LSTM(50, return_sequences=False, input_shape=(lookback, 1)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, epochs=200, batch_size=16, verbose=0)

# Forecast for 2024 and 2025
forecast_scaled = []
input_seq = scaled_data[-lookback:].reshape(1, lookback, 1)

for _ in range(2):  # Forecast 2 steps ahead
    pred = model.predict(input_seq, verbose=0)[0][0]
    forecast_scaled.append(pred)
    input_seq = np.append(input_seq[:, 1:, :], [[[pred]]], axis=1)

# Inverse transform forecasted values
forecast = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# Plotting
plt.figure(figsize=(10, 5))

# Actual data
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

# Forecasted values
plt.plot(forecast_years, forecast, marker='o', linestyle='--', color='green', label='LSTM Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='green')

plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

plt.title("LSTM Forecast of Remaining Forest Percentage in Modhupur (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual
print("\nLSTM Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ----------------------------------------
# Actual data from 2006 to 2025
# ----------------------------------------
years = list(range(2006, 2026))
remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]

# Training & test split
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# ----------------------------------------
# Instead of LSTM, use given forecast values
# ----------------------------------------
# Commented out LSTM section
# forecast = ... (predicted by LSTM)
forecast = [18.85, 18.86]   # Manual values provided
forecast_years = [2024, 2025]

# ----------------------------------------
# Apply Clay Style
# ----------------------------------------
plt.rcParams['legend.handlelength'] = 0
plt.rcParams['legend.numpoints'] = 1
plt.rcParams['lines.linewidth'] = 2
plt.rcParams["font.family"] = "Arial"
plt.rcParams['font.size'] = 20
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Arial"
plt.rcParams["mathtext.it"] = "Arial:italic"
plt.rcParams["mathtext.bf"] = "Arial:bold"

fig, ax = plt.subplots(figsize=(8, 6))

# ----------------------------------------
# Plot actual data
# ----------------------------------------
ax.plot(years, remaining_forest_percentages, '-o', color='blue', label='Actual (2006–2025)', markersize=6)
for x, y_val in zip(years, remaining_forest_percentages):
    ax.text(x, y_val - 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=10, color='blue')

# Plot forecasted values
ax.plot(forecast_years, forecast, '--o', color='green', label='LSTM Forecast (2024–2025)', markersize=6)
for x, y_val in zip(forecast_years, forecast):
    ax.text(x, y_val + 0.6, f'{y_val:.2f}', ha='center', va='top', fontsize=10, color='green')

# Vertical line marking forecast start
ax.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

# ----------------------------------------
# Axis labels
# ----------------------------------------
ax.set_xlabel("Year", fontsize=25)
ax.set_ylabel("Forest Percentage (\%)", fontsize=25)

# X-axis ticks
ax.set_xticks(range(2006, 2026))
plt.xticks(rotation=45)

# Customize ticks (outward)
ax.tick_params(axis='both', which='major', direction='out', length=6, width=2, colors='black', labelsize=20)
ax.tick_params(axis='both', which='minor', direction='out', length=4, width=2, colors='black', labelsize=20)

# Major/minor ticks
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))   # every 2 years
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))   # every 1 percent
ax.minorticks_on()
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))   # every 1 year minor
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.5)) # every 0.5 percent minor

# Thicken axis lines
for spine in ax.spines.values():
    spine.set_linewidth(2)

# Legend
ax.legend(frameon=False, fontsize=20)

# Tight layout & show
plt.tight_layout()
plt.show()

# ----------------------------------------
# Print forecast vs actual
# ----------------------------------------
print("\nLSTM Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Flatten, Dense

# Actual data from 2006 to 2025
years = list(range(2006, 2026))
remaining_forest_percentages = [
      21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]

# Training data up to 2023
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# Normalize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# Prepare sequences (lookback = 3)
lookback = 8
X, y = [], []
for i in range(lookback, len(scaled_data)):
    X.append(scaled_data[i - lookback:i])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)

# Reshape to 5D for ConvLSTM2D: (samples, time steps, rows, cols, channels)
X = X.reshape((X.shape[0], lookback, 1, 1, 1))

# Build ConvLSTM model
model = Sequential([
    ConvLSTM2D(filters=32, kernel_size=(1, 1), activation='relu',
               input_shape=(lookback, 1, 1, 1), return_sequences=False),
    Flatten(),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, epochs=200, batch_size=16, verbose=0)

# Forecast for 2024 and 2025
forecast_scaled = []
last_sequence = scaled_data[-lookback:].reshape((1, lookback, 1, 1, 1))

for _ in range(2):
    pred = model.predict(last_sequence, verbose=0)[0][0]
    forecast_scaled.append(pred)
    # Update sequence with the new prediction
    new_step = np.array(pred).reshape(1, 1, 1, 1, 1)
    last_sequence = np.concatenate((last_sequence[:, 1:], new_step), axis=1)

# Inverse transform forecasted values
forecast = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# Plotting
plt.figure(figsize=(10, 5))

# Actual data
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

# Forecasted values
plt.plot(forecast_years, forecast, marker='o', linestyle='--', color='purple', label='ConvLSTM Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='purple')

plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

plt.title("ConvLSTM Forecast of Remaining Forest Percentage in Modhupur (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual
print("\nConvLSTM Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

# Actual data from 2006 to 2025chtg
years = list(range(2006, 2026))
remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]

# Training data up to 2023
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# Normalize the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# Prepare sequences (lookback = 8), direct multi-step prediction (2 years ahead)
lookback = 8
forecast_horizon = 2
X, y = [], []
for i in range(lookback, len(scaled_data) - forecast_horizon + 1):
    X.append(scaled_data[i - lookback:i])
    y.append(scaled_data[i:i + forecast_horizon, 0])  # two years ahead

X, y = np.array(X), np.array(y)

# Reshape to 5D for ConvLSTM2D: (samples, time steps, rows, cols, channels)
X = X.reshape((X.shape[0], lookback, 1, 1, 1))

# Build ConvLSTM model
model = Sequential([
    ConvLSTM2D(filters=64, kernel_size=(1, 1), activation='relu',
               input_shape=(lookback, 1, 1, 1), return_sequences=False),
    Flatten(),
    Dense(50, activation='relu'),
    Dense(forecast_horizon)  # output = 2 years at once
])
model.compile(optimizer=Adam(learning_rate=1e-4), loss='mean_squared_error')

# Train
model.fit(X, y, epochs=300, batch_size=16, verbose=0)

# Forecast for 2024–2025
last_sequence = scaled_data[-lookback:].reshape((1, lookback, 1, 1, 1))
forecast_scaled = model.predict(last_sequence, verbose=0).flatten()

# Inverse transform forecasted values
forecast = scaler.inverse_transform(forecast_scaled.reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# Plotting
plt.figure(figsize=(10, 5))

# Actual data
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

# Forecasted values
plt.plot(forecast_years, forecast, marker='o', linestyle='--', color='red', label='ConvLSTM Direct Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='red')

plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')

plt.title("ConvLSTM Direct Multi-step Forecast of Remaining Forest Percentage in Madhupur (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual
print("\nConvLSTM Direct Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, ConvLSTM2D, Flatten
from tensorflow.keras.regularizers import L1L2
from sklearn.metrics import mean_squared_error

# Actual data from 2006 to 2025dpsssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss
years = list(range(2006, 2026))
remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]

# Training data up to 2023
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]  # Actual values for 2024–2025

# 1. Try different lookback periods
lookback_options = [3, 5, 8, 10]
best_lookback = 8
best_mse = float('inf')

for lookback in lookback_options:
    # Normalize the data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))
    
    # Prepare sequences
    X, y = [], []
    for i in range(lookback, len(scaled_data)):
        X.append(scaled_data[i - lookback:i])
        y.append(scaled_data[i, 0])
    
    X, y = np.array(X), np.array(y)
    
    # Reshape for ConvLSTM2D
    X_conv = X.reshape((X.shape[0], lookback, 1, 1, 1))
    
    # Build and train model
    model = Sequential([
        ConvLSTM2D(filters=32, kernel_size=(1, 1), activation='relu',
                   input_shape=(lookback, 1, 1, 1), return_sequences=False),
        Flatten(),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X_conv, y, epochs=200, batch_size=1, verbose=0, validation_split=0.2)
    
    # Evaluate on validation set
    val_loss = model.evaluate(X_conv, y, verbose=0)
    
    if val_loss < best_mse:
        best_mse = val_loss
        best_lookback = lookback

print(f"Best lookback period: {best_lookback}")

# 2. Try different model architectures
# Option A: Standard LSTM (often better for univariate time series)
def create_lstm_model(lookback):
    model = Sequential([
        LSTM(50, activation='relu', input_shape=(lookback, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# Option B: More complex ConvLSTM
def create_complex_convlstm(lookback):
    model = Sequential([
        ConvLSTM2D(filters=64, kernel_size=(1, 1), activation='relu',
                   input_shape=(lookback, 1, 1, 1), return_sequences=True),
        ConvLSTM2D(filters=32, kernel_size=(1, 1), activation='relu', return_sequences=False),
        Flatten(),
        Dense(50, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

# 3. Try different scaling ranges
scaler_options = [
    MinMaxScaler(feature_range=(0, 1)),
    MinMaxScaler(feature_range=(-1, 1))
]

# 4. Try different training parameters
epoch_options = [100, 200, 300]
batch_options = [1, 2, 4]

# Implementation with selected parameters
lookback = best_lookback  # Or manually set based on analysis
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# Prepare sequences
X, y = [], []
for i in range(lookback, len(scaled_data)):
    X.append(scaled_data[i - lookback:i])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)

# Choose model architecture (try different ones)
model = create_lstm_model(lookback)  # Try this for potentially better results
# model = create_complex_convlstm(lookback)  # Or try this

# Reshape data based on model choice
if isinstance(model.layers[0], LSTM):
    X_reshaped = X.reshape((X.shape[0], X.shape[1], 1))
else:
    X_reshaped = X.reshape((X.shape[0], lookback, 1, 1, 1))

# Train with more epochs and different batch size
history = model.fit(X_reshaped, y, epochs=300, batch_size=16, verbose=0, validation_split=0.2)

# Forecast for 2024 and 2025
forecast_scaled = []
if isinstance(model.layers[0], LSTM):
    last_sequence = scaled_data[-lookback:].reshape((1, lookback, 1))
else:
    last_sequence = scaled_data[-lookback:].reshape((1, lookback, 1, 1, 1))

for _ in range(2):
    pred = model.predict(last_sequence, verbose=0)[0][0]
    forecast_scaled.append(pred)
    
    # Update sequence with the new prediction
    if isinstance(model.layers[0], LSTM):
        new_step = np.array(pred).reshape(1, 1, 1)
        last_sequence = np.concatenate((last_sequence[:, 1:], new_step), axis=1)
    else:
        new_step = np.array(pred).reshape(1, 1, 1, 1, 1)
        last_sequence = np.concatenate((last_sequence[:, 1:], new_step), axis=1)

# Inverse transform forecasted values
forecast = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

plt.plot(forecast_years, forecast, marker='o', linestyle='--', color='purple', label='Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='purple')

plt.axvline(x=2024, linestyle='--', color='gray', label='Forecast Start')
plt.title("ConvLSTM Forecast of Remaining Forest Percentage in Modhupur (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Print forecast vs actual
print("\nImproved Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    comment = "very close!" if diff < 0.05 else "a small gap"
    print(f"{year}: Forecast = {predicted:.2f}%, Actual = {actual:.2f}%, "
          f"Difference = {diff:.2f}% ({comment})")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ========================================
# 1. Load Data
# ========================================
years = list(range(2006, 2026))
remaining_forest_percentages = [
    21.18, 21.42, 21.46, 22.19, 21.50, 21.51, 22.44,
    20.07, 18.78, 18.17, 19.11, 18.71, 19.25,
    18.69, 18.96, 18.81, 18.87, 18.91, 18.92, 18.42
]

# Split: train up to 2023, test = 2024–2025
training_years = list(range(2006, 2024))
training_data = remaining_forest_percentages[:len(training_years)]
test_data = remaining_forest_percentages[len(training_years):]

# ========================================
# 2. Normalize Data
# ========================================
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(np.array(training_data).reshape(-1, 1))

# ========================================
# 3. Prepare Sequences
# ========================================
lookback = 8  # Use last 8 years to predict next
X, y = [], []

for i in range(lookback, len(scaled_data)):
    # Shape: (time_steps, rows, cols, channels)
    X.append(scaled_data[i - lookback:i].reshape(lookback, 1, 1, 1))
    y.append(scaled_data[i, 0])

X = np.array(X)
y = np.array(y)

print(f"Input shape: {X.shape}")  # Should be (samples, lookback, 1, 1, 1)

# ========================================
# 4. Build Improved ConvLSTM2D Model
# ========================================
model = Sequential([
    ConvLSTM2D(
        filters=64,
        kernel_size=(1, 1),              # Fixed: now compatible
        activation='relu',
        input_shape=(lookback, 1, 1, 1),
        return_sequences=True,
        dropout=0.2
    ),
    ConvLSTM2D(
        filters=32,
        kernel_size=(1, 1),
        activation='relu',
        return_sequences=False,
        dropout=0.2
    ),
    Flatten(),
    Dense(16, activation='relu'),
    Dropout(0.1),
    Dense(1)
])

model.compile(optimizer='adam', loss='mae', metrics=['mae'])
model.fit(X, y, epochs=200, batch_size=16, verbose=0)

# Show model summary
model.summary()

# ========================================
# 5. Train with Early Stopping
# ========================================
early_stop = EarlyStopping(monitor='loss', patience=15, restore_best_weights=True)

history = model.fit(
    X, y,
    epochs=300,
    batch_size=16,
    verbose=0,
    callbacks=[early_stop]
)

# Optional: Plot training loss
plt.figure(figsize=(8, 3))
plt.plot(history.history['loss'])
plt.title('Model Training Loss (MAE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

# ========================================
# 6. Forecast 2024 and 2025 (Recursive)
# ========================================
forecast_scaled = []
last_sequence = scaled_data[-lookback:].reshape(1, lookback, 1, 1, 1)

for _ in range(2):
    pred = model.predict(last_sequence, verbose=0)[0][0]
    forecast_scaled.append(pred)
    
    # Update sequence: shift left, append new prediction
    new_step = np.array(pred).reshape(1, 1, 1, 1, 1)
    last_sequence = np.concatenate([last_sequence[:, 1:, :, :, :], new_step], axis=1)

# Inverse transform to original scale
forecast = scaler.inverse_transform(np.array(forecast_scaled).reshape(-1, 1)).flatten()
forecast_years = [2024, 2025]

# ========================================
# 7. Plot Results
# ========================================
plt.figure(figsize=(10, 5))

# Actual data (2006–2025)
plt.plot(years, remaining_forest_percentages, marker='o', color='blue', label='Actual (2006–2025)')
for x, y_val in zip(years, remaining_forest_percentages):
    plt.text(x, y_val - 0.4, f'{y_val:.2f}', ha='center', va='bottom', fontsize=8, color='blue')

# Forecast
plt.plot(forecast_years, forecast, marker='s', linestyle='--', color='red', label='ConvLSTM Forecast (2024–2025)')
for x, y_val in zip(forecast_years, forecast):
    plt.text(x, y_val + 0.3, f'{y_val:.2f}', ha='center', va='top', fontsize=8, color='red')

# Forecast start line
plt.axvline(x=2024, linestyle='--', color='gray', alpha=0.7, label='Forecast Start')

# Labels and formatting
plt.title("Improved ConvLSTM Forecast of Forest Cover in Madhupur (2006–2025)")
plt.xlabel("Year")
plt.ylabel("Forest Percentage (%)")
plt.xticks(ticks=range(2006, 2026), rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# ========================================
# 8. Print Forecast vs Actual
# ========================================
print("\n📈 ConvLSTM Forecast vs Actual (2024–2025):")
for i, year in enumerate(forecast_years):
    predicted = forecast[i]
    actual = test_data[i]
    diff = abs(predicted - actual)
    status = "✅ Very close!" if diff < 0.1 else "⚠️  Moderate gap"
    print(f"{year}: Forecast = {predicted:.2f}% | Actual = {actual:.2f}% | Diff = {diff:.2f}% → {status}")